In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

from datetime import datetime
from shapely.geometry import Point

pd.set_option('display.max_columns', None)

In [3]:
dt = pd.read_csv('C:/Users/sigmalburg/Documents/МНАД I курс/Пожары!/data_fires_128_form_1T_v20240912.csv',sep=';')
dt.head()

C:\Users\sigmalburg\AppData\Local\Temp\ipykernel_23444\2508921810.py:1: DtypeWarning: Columns (0: comment, 1: zone) have mixed types. Specify dtype option on import or set low_memory=False.
  dt = pd.read_csv('C:/Users/sigmalburg/Documents/МНАД I курс/Пожары!/data_fires_128_form_1T_v20240912.csv',sep=';')


,region,oktmo,okato,year,type,code,latitude,longitude,zone_beginning,landmark_settlement,landmark_azimuth,landmark_distance,forestry,date_beginning,area_beginning,date_end,current_state,area_total,area_forest,area_fund_total,area_fund_forest,comment,zone
0,Белгородская область,14000000,14000000,2000,Лесные,к-12 (15254),50.513889,36.653056,Наземный мониторинг,н.п. Разумное,212.0,2.4,Белгородское,01.05.2000,48,01.05.2000,ликвидирован 11.05.2000,48,24,0,0,NaN,NaN
1,Белгородская область,14000000,14000000,2000,Лесные,к-15 (16171),50.911944,37.998889,Наземный мониторинг,н.п. Нов.Безгинка,282.0,8.8,Чернянское,04.05.2000,48,04.05.2000,ликвидирован 14.05.2000,48,11,0,0,NaN,NaN
2,Белгородская область,14000000,14000000,2000,Лесные,к-10 (35227),51.053889,36.016944,Наземный мониторинг,н.п. Пены,105.0,7.2,Ивнянское,22.08.2000,48,22.08.2000,ликвидирован 1.09.2000,46,3,0,0,NaN,NaN
3,Белгородская область,14000000,14000000,2000,Лесные,к-48 (35489),50.631944,37.048056,Наземный мониторинг,н.п. Большое Городище,286.0,1.9,Шебекинское,23.08.2000,48,23.08.2000,ликвидирован 2.09.2000,48,10,0,0,NaN,NaN
4,Белгородская область,14000000,14000000,2000,Лесные,к-20 (37323),51.316944,38.233889,Наземный мониторинг,н.п. Старомеловое,214.0,4.8,Горшенченское,03.09.2000,48,03.09.2000,ликвидирован 13.09.2000,5,0,0,0,NaN,NaN


In [4]:
dt['date_liquidation'] = dt['current_state'].str.extract(r'(\d{1,2}.\d{1,2}.\d{4})')

dt['date_beginning'] = pd.to_datetime(dt['date_beginning'],dayfirst=True)
dt['date_end'] = pd.to_datetime(dt['date_end'],dayfirst=True)
dt['date_liquidation'] = pd.to_datetime(dt['date_liquidation'],dayfirst=True)

dt['date_end_max'] = dt[['date_end','date_liquidation']].max(axis=1)
dt['duration'] = (dt['date_end_max'] - dt['date_beginning']).dt.days

In [9]:
dt.to_csv('C:/Users/sigmalburg/Documents/МНАД I курс/Пожары!/data_fires.csv', index=False)

In [6]:
dtw_headers = pd.read_table('C:/Users/sigmalburg/Documents/МНАД I курс/\Пожары!/wr377097a4/fld377097a4.txt',encoding='cp1251',sep='\t',header=None)
dtw_stations= pd.read_table('C:/Users/sigmalburg/Documents/МНАД I курс/\Пожары!/wr377097a4/statlist377097a4.txt',encoding='cp1251',sep='\t',header=None)
dtw = pd.read_csv('C:/Users/sigmalburg/Documents/МНАД I курс/\Пожары!/wr377097a4/wr377097a4.txt',sep=';',header=None)

C:\Users\sigmalburg\AppData\Local\Temp\ipykernel_23444\2676045937.py:3: DtypeWarning: Columns (0: 5, 1: 7, 2: 9, 3: 11, 4: 13, 5: 16, 6: 19, 7: 21, 8: 23, 9: 25) have mixed types. Specify dtype option on import or set low_memory=False.
  dtw = pd.read_csv('C:/Users/sigmalburg/Documents/МНАД I курс/\Пожары!/wr377097a4/wr377097a4.txt',sep=';',header=None)


In [7]:
dtw.columns = dtw_headers[3]
# удаляем признаки качества - считаем все данные проверенными
dtw = dtw.loc[:, ~dtw.columns.duplicated()]
dtw = dtw.drop(columns=['Признак качества','Погода в срок наблюдения','Признак наличия знака >','Температура поверхности почвы'])

In [8]:
numeric_cols = [
    'Общее количество облачности',
    'Средняя скорость ветра',
    'Максимальная скорость ветра',
    'Сумма осадков',
    'Мин.температура воздуха между сроками',
    'Макс. темперура воздуха между сроками'
]

for col in numeric_cols:
    dtw[col] = pd.to_numeric(dtw[col], errors='coerce')

category_cols = [
    'Погода между сроками',
    'Направление ветра'
]

In [9]:
# словарь агрегаций: для чисел — mean, для категорий — мода

def get_mode(x):
    m = x.mode()
    return m.iloc[0] if len(m) > 0 else None

agg_dict = {col: 'mean' for col in numeric_cols}
agg_dict.update({col: get_mode for col in category_cols})

daily = (
    dtw
    .groupby(['Синоптический индекс станции', 'Год   по Гринвичу', 'Месяц по Гринвичу', 'День  по Гринвичу'], as_index=False)
    .agg(agg_dict)
)

In [10]:
daily['Средняя температура воздуха'] = (daily['Макс. темперура воздуха между сроками'] + daily['Мин.температура воздуха между сроками']) / 2
daily.head()

3,Синоптический индекс станции,Год по Гринвичу,Месяц по Гринвичу,День по Гринвичу,Общее количество облачности,Средняя скорость ветра,Максимальная скорость ветра,Сумма осадков,Мин.температура воздуха между сроками,Макс. темперура воздуха между сроками,Погода между сроками,Направление ветра,Средняя температура воздуха
0,20674,2000,1,1,5.625,8.625,12.000,0.0,-28.8125,-27.4250,3,192,-28.11875
1,20674,2000,1,2,2.125,6.000,9.875,0.0,-33.1625,-32.4625,3,172,-32.81250
2,20674,2000,1,3,0.625,7.500,11.625,0.0,-32.3500,-31.2375,3,187,-31.79375
3,20674,2000,1,4,0.000,5.750,9.375,0.0,-33.1250,-32.3750,0,175,-32.75000
4,20674,2000,1,5,0.000,5.750,8.875,0.0,-32.8375,-32.2125,3,187,-32.52500


In [ ]:
daily = pd.merge(daily, dtw_stations, how = "left", left_on = 'Синоптический индекс станции', right_on = dtw_stations.columns[0])
daily.to_csv('C:/Users/sigmalburg/Documents/МНАД I курс/Пожары!/data_weather.csv', index=False)

In [24]:
stations = pd.read_csv('C:/Users/sigmalburg/Documents/МНАД I курс/\Пожары!/ghcnd-stations.csv',sep=';',encoding='cp1251')

In [ ]:
daily_coord = pd.merge(daily,stations,how='left',left_on='Синоптический индекс станции',right_on='Station ID')
daily_coord = daily_coord[daily_coord['Station ID'].isna()==False]

,Синоптический индекс станции,Год по Гринвичу,Месяц по Гринвичу,День по Гринвичу,Общее количество облачности,Средняя скорость ветра,Максимальная скорость ветра,Сумма осадков,Мин.температура воздуха между сроками,Макс. темперура воздуха между сроками,Погода между сроками,Направление ветра,Средняя температура воздуха,0,1,Station Code,latitude,longitude,elevation,Station name,system,Station ID,State
0,20674,2000,1,1,5.625,8.625,12.000,0.0000,-28.8125,-27.4250,3,192,-28.11875,20674,Диксон,RSM00020674,73.5000,80.40,42.0,DIKSON,GSN,20674.0,RS
1,20674,2000,1,2,2.125,6.000,9.875,0.0000,-33.1625,-32.4625,3,172,-32.81250,20674,Диксон,RSM00020674,73.5000,80.40,42.0,DIKSON,GSN,20674.0,RS
2,20674,2000,1,3,0.625,7.500,11.625,0.0000,-32.3500,-31.2375,3,187,-31.79375,20674,Диксон,RSM00020674,73.5000,80.40,42.0,DIKSON,GSN,20674.0,RS
3,20674,2000,1,4,0.000,5.750,9.375,0.0000,-33.1250,-32.3750,0,175,-32.75000,20674,Диксон,RSM00020674,73.5000,80.40,42.0,DIKSON,GSN,20674.0,RS
4,20674,2000,1,5,0.000,5.750,8.875,0.0000,-32.8375,-32.2125,3,187,-32.52500,20674,Диксон,RSM00020674,73.5000,80.40,42.0,DIKSON,GSN,20674.0,RS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1206150,37472,2024,12,27,10.000,2.750,6.250,0.3625,3.0250,3.5125,6,301,3.26875,37472,Махачкала,RSM00037472,42.9667,47.55,28.0,MAHACKALA,NaN,37472.0,RS
1206151,37472,2024,12,28,10.000,4.500,9.500,0.1500,2.3250,2.9625,2,297,2.64375,37472,Махачкала,RSM00037472,42.9667,47.55,28.0,MAHACKALA,NaN,37472.0,RS
1206152,37472,2024,12,29,9.750,4.500,11.500,0.0375,0.2500,1.1000,2,295,0.67500,37472,Махачкала,RSM00037472,42.9667,47.55,28.0,MAHACKALA,NaN,37472.0,RS
1206153,37472,2024,12,30,10.000,1.375,4.875,0.0000,-0.7250,-0.1625,2,0,-0.44375,37472,Махачкала,RSM00037472,42.9667,47.55,28.0,MAHACKALA,NaN,37472.0,RS


In [26]:
# одной станции не нашлось в массиве Global Historical Climatology Network daily (GHCNd)
# https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt
# это 32586 - Петропавловский маяк, мыс Маячный (Елизовский район, Камчатский край)
# удалим ее из датасета - 9132 строк из 1206155

In [28]:
# делаем GeoDataFrame из пожаров
gdf_fires = gpd.GeoDataFrame(
    dt,
    geometry=gpd.points_from_xy(dt['longitude'], dt['latitude']),
    crs="EPSG:4326"  # WGS84 — стандартный формат координат
)

# Делаем GeoDataFrame из станций
gdf_stations = gpd.GeoDataFrame(
    daily_coord,
    geometry=gpd.points_from_xy(daily_coord['longitude'], daily_coord['latitude']),
    crs="EPSG:4326"
)

# известно, что координаты станций не совсем совпадают с реальностью, по крайней мере, в этой С.К. Оставим на будущее! 

In [ ]:
gdf_stations = gdf_stations[['Синоптический индекс станции','latitude','longitude','Station name','geometry']].drop_duplicates()
gdf_stations.head()

,Синоптический индекс станции,latitude,longitude,Station name,geometry
0,20674,73.5000,80.4000,DIKSON,POINT (80.4 73.5)
9132,20891,71.9831,102.4667,HATANGA,POINT (102.4667 71.9831)
18264,21946,70.6167,147.8831,CHOKURDAH,POINT (147.8831 70.6167)
27396,21982,70.9831,-178.4833,OSTROV VRANGELJA,POINT (-178.4833 70.9831)
36528,22113,68.9667,33.0497,MURMANSK,POINT (33.0497 68.9667)
...,...,...,...,...,...
1160495,36034,51.5000,81.2167,RUBCOVSK,POINT (81.2167 51.5)
1169627,37031,44.9831,41.1167,ARMAVIR,POINT (41.1167 44.9831)
1178759,37099,43.5800,39.7700,SOTCHI,POINT (39.77 43.58)
1187891,37235,43.3500,45.6831,GROZNYJ,POINT (45.6831 43.35)


In [ ]:
joined = gpd.sjoin_nearest(
    gdf_fires,
    gdf_stations,
    how='left',           # каждому пожару присваивается ближайшая станция по координатам (или NaN, если станций нет)
    distance_col='distance_m'  # добавим колонку с расстоянием до станции В ГРАДУСАХ
)
joined.head()

c:\Users\sigmalburg\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


,region,oktmo,okato,year,type,code,latitude_left,longitude_left,zone_beginning,landmark_settlement,landmark_azimuth,landmark_distance,forestry,date_beginning,area_beginning,date_end,current_state,area_total,area_forest,area_fund_total,area_fund_forest,comment,zone,date_liquidation,date_end_max,duration,geometry,index_right,Синоптический индекс станции,latitude_right,longitude_right,Station name,distance_m
0,Белгородская область,14000000,14000000,2000,Лесные,к-12 (15254),50.513889,36.653056,Наземный мониторинг,н.п. Разумное,212.0,2.4,Белгородское,2000-05-01,48,2000-05-01,ликвидирован 11.05.2000,48,24,0,0,NaN,NaN,2000-05-11,2000-05-11,10,POINT (36.65306 50.51389),1060043,34009,51.7667,36.1667,KURSK,1.343904
1,Белгородская область,14000000,14000000,2000,Лесные,к-15 (16171),50.911944,37.998889,Наземный мониторинг,н.п. Нов.Безгинка,282.0,8.8,Чернянское,2000-05-04,48,2000-05-04,ликвидирован 14.05.2000,48,11,0,0,NaN,NaN,2000-05-14,2000-05-14,10,POINT (37.99889 50.91194),1069175,34123,51.7000,39.2167,VORONEZ,1.450550
2,Белгородская область,14000000,14000000,2000,Лесные,к-10 (35227),51.053889,36.016944,Наземный мониторинг,н.п. Пены,105.0,7.2,Ивнянское,2000-08-22,48,2000-08-22,ликвидирован 1.09.2000,46,3,0,0,NaN,NaN,2000-09-01,2000-09-01,10,POINT (36.01694 51.05389),1060043,34009,51.7667,36.1667,KURSK,0.728372
3,Белгородская область,14000000,14000000,2000,Лесные,к-48 (35489),50.631944,37.048056,Наземный мониторинг,н.п. Большое Городище,286.0,1.9,Шебекинское,2000-08-23,48,2000-08-23,ликвидирован 2.09.2000,48,10,0,0,NaN,NaN,2000-09-02,2000-09-02,10,POINT (37.04806 50.63194),1060043,34009,51.7667,36.1667,KURSK,1.436822
4,Белгородская область,14000000,14000000,2000,Лесные,к-20 (37323),51.316944,38.233889,Наземный мониторинг,н.п. Старомеловое,214.0,4.8,Горшенченское,2000-09-03,48,2000-09-03,ликвидирован 13.09.2000,5,0,0,0,NaN,NaN,2000-09-13,2000-09-13,10,POINT (38.23389 51.31694),1069175,34123,51.7000,39.2167,VORONEZ,1.054822
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1459459,Чеченская Республика,96000000,96000000,2024,Нелесные,к-305 (86309),43.178056,45.233889,Земли иных категорий,н.п. Бамут,52.0,3.9,Ассиновское,2024-09-04,3,2024-09-04,ликвидирован 9.09.2024,7,0,0,0,NaN,Земли иных категорий,2024-09-09,2024-09-09,5,POINT (45.23389 43.17806),1187891,37235,43.3500,45.6831,GROZNYJ,0.480994
1459460,Чеченская Республика,96000000,96000000,2024,Нелесные,к-306 (88159),43.296111,45.626944,Земли иных категорий,н.п. Грозный,246.0,3.8,Грозненское,2024-09-09,2,2024-09-09,действует,3,0,0,0,NaN,Земли иных категорий,NaT,2024-09-09,0,POINT (45.62694 43.29611),1187891,37235,43.3500,45.6831,GROZNYJ,0.077830
1459461,Чеченская Республика,96000000,96000000,2024,Нелесные,к-307 (88186),43.311944,46.068056,Земли иных категорий,н.п. Белоречье,321.0,4.4,Гудермесское,2024-09-09,2,2024-09-11,действует,7,0,0,0,NaN,Земли иных категорий,NaT,2024-09-11,2,POINT (46.06806 43.31194),1187891,37235,43.3500,45.6831,GROZNYJ,0.386832
1459462,Чеченская Республика,96000000,96000000,2024,Нелесные,к-308 (88598),43.213056,45.845000,Земли иных категорий,н.п. Чечен-аул,66.0,5.1,Шалинское,2024-09-11,2,2024-09-11,действует,10,0,0,0,NaN,Земли иных категорий,NaT,2024-09-11,0,POINT (45.845 43.21306),1187891,37235,43.3500,45.6831,GROZNYJ,0.212050


In [39]:
daily_coord['date'] = pd.to_datetime(daily_coord[['Год   по Гринвичу','Месяц по Гринвичу','День  по Гринвичу']].astype(str).agg('-'.join, axis=1))

In [41]:
# скользящие окна для приближения погоды до пожара

weather = daily_coord.sort_values(['Синоптический индекс станции', 'date'])

# Окна
w = weather
w['temp_3d'] = w.groupby('Синоптический индекс станции')['Средняя температура воздуха'].transform(lambda x: x.rolling(3, min_periods=1).mean())
w['wind_3d_max'] = w.groupby('Синоптический индекс станции')['Средняя скорость ветра'].transform(lambda x: x.rolling(3, min_periods=1).max())
w['rain_7d_sum'] = w.groupby('Синоптический индекс станции')['Сумма осадков'].transform(lambda x: x.rolling(7, min_periods=1).sum())

# Сдвигаем на 1, чтобы получить «до»
w['temp_3d_lag'] = w.groupby('Синоптический индекс станции')['temp_3d'].shift(1)
w['wind_3d_max_lag'] = w.groupby('Синоптический индекс станции')['wind_3d_max'].shift(1)
w['rain_7d_sum_lag'] = w.groupby('Синоптический индекс станции')['rain_7d_sum'].shift(1)

# Сохраняем только нужные колонки
weather_features = w[['Синоптический индекс станции', 'date', 'temp_3d_lag', 'wind_3d_max_lag', 'rain_7d_sum_lag']]

# Дальше merge с пожарами по station_id и start_date

In [ ]:
# итоговый датасет

merged_df = pd.merge(
    joined, 
    weather_features, 
    left_on=['Синоптический индекс станции', 'date_beginning'],
    right_on = ['Синоптический индекс станции', 'date'],
    how='left'
)

,region,oktmo,okato,year,type,code,latitude_left,longitude_left,zone_beginning,landmark_settlement,landmark_azimuth,landmark_distance,forestry,date_beginning,area_beginning,date_end,current_state,area_total,area_forest,area_fund_total,area_fund_forest,comment,zone,date_liquidation,date_end_max,duration,geometry,index_right,Синоптический индекс станции,latitude_right,longitude_right,Station name,distance_m,date,temp_3d_lag,wind_3d_max_lag,rain_7d_sum_lag
0,Белгородская область,14000000,14000000,2000,Лесные,к-12 (15254),50.513889,36.653056,Наземный мониторинг,н.п. Разумное,212.0,2.4,Белгородское,2000-05-01,48,2000-05-01,ликвидирован 11.05.2000,48,24,0,0,NaN,NaN,2000-05-11,2000-05-11,10,POINT (36.65306 50.51389),1060043,34009,51.7667,36.1667,KURSK,1.343904,2000-05-01,14.477083,4.125,0.4000
1,Белгородская область,14000000,14000000,2000,Лесные,к-15 (16171),50.911944,37.998889,Наземный мониторинг,н.п. Нов.Безгинка,282.0,8.8,Чернянское,2000-05-04,48,2000-05-04,ликвидирован 14.05.2000,48,11,0,0,NaN,NaN,2000-05-14,2000-05-14,10,POINT (37.99889 50.91194),1069175,34123,51.7000,39.2167,VORONEZ,1.450550,2000-05-04,4.552083,4.750,0.1375
2,Белгородская область,14000000,14000000,2000,Лесные,к-10 (35227),51.053889,36.016944,Наземный мониторинг,н.п. Пены,105.0,7.2,Ивнянское,2000-08-22,48,2000-08-22,ликвидирован 1.09.2000,46,3,0,0,NaN,NaN,2000-09-01,2000-09-01,10,POINT (36.01694 51.05389),1060043,34009,51.7667,36.1667,KURSK,0.728372,2000-08-22,21.981250,2.250,0.0750
3,Белгородская область,14000000,14000000,2000,Лесные,к-48 (35489),50.631944,37.048056,Наземный мониторинг,н.п. Большое Городище,286.0,1.9,Шебекинское,2000-08-23,48,2000-08-23,ликвидирован 2.09.2000,48,10,0,0,NaN,NaN,2000-09-02,2000-09-02,10,POINT (37.04806 50.63194),1060043,34009,51.7667,36.1667,KURSK,1.436822,2000-08-23,24.158333,3.000,0.0750
4,Белгородская область,14000000,14000000,2000,Лесные,к-20 (37323),51.316944,38.233889,Наземный мониторинг,н.п. Старомеловое,214.0,4.8,Горшенченское,2000-09-03,48,2000-09-03,ликвидирован 13.09.2000,5,0,0,0,NaN,NaN,2000-09-13,2000-09-13,10,POINT (38.23389 51.31694),1069175,34123,51.7000,39.2167,VORONEZ,1.054822,2000-09-03,18.406250,2.375,0.1000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1459459,Чеченская Республика,96000000,96000000,2024,Нелесные,к-305 (86309),43.178056,45.233889,Земли иных категорий,н.п. Бамут,52.0,3.9,Ассиновское,2024-09-04,3,2024-09-04,ликвидирован 9.09.2024,7,0,0,0,NaN,Земли иных категорий,2024-09-09,2024-09-09,5,POINT (45.23389 43.17806),1187891,37235,43.3500,45.6831,GROZNYJ,0.480994,2024-09-04,24.891667,1.125,0.0000
1459460,Чеченская Республика,96000000,96000000,2024,Нелесные,к-306 (88159),43.296111,45.626944,Земли иных категорий,н.п. Грозный,246.0,3.8,Грозненское,2024-09-09,2,2024-09-09,действует,3,0,0,0,NaN,Земли иных категорий,NaT,2024-09-09,0,POINT (45.62694 43.29611),1187891,37235,43.3500,45.6831,GROZNYJ,0.077830,2024-09-09,18.906250,1.750,3.9000
1459461,Чеченская Республика,96000000,96000000,2024,Нелесные,к-307 (88186),43.311944,46.068056,Земли иных категорий,н.п. Белоречье,321.0,4.4,Гудермесское,2024-09-09,2,2024-09-11,действует,7,0,0,0,NaN,Земли иных категорий,NaT,2024-09-11,2,POINT (46.06806 43.31194),1187891,37235,43.3500,45.6831,GROZNYJ,0.386832,2024-09-09,18.906250,1.750,3.9000
1459462,Чеченская Республика,96000000,96000000,2024,Нелесные,к-308 (88598),43.213056,45.845000,Земли иных категорий,н.п. Чечен-аул,66.0,5.1,Шалинское,2024-09-11,2,2024-09-11,действует,10,0,0,0,NaN,Земли иных категорий,NaT,2024-09-11,0,POINT (45.845 43.21306),1187891,37235,43.3500,45.6831,GROZNYJ,0.212050,2024-09-11,20.287500,1.000,3.9000


In [48]:
merged_df.head()

,region,oktmo,okato,year,type,code,latitude_left,longitude_left,zone_beginning,landmark_settlement,landmark_azimuth,landmark_distance,forestry,date_beginning,area_beginning,date_end,current_state,area_total,area_forest,area_fund_total,area_fund_forest,comment,zone,date_liquidation,date_end_max,duration,geometry,index_right,Синоптический индекс станции,latitude_right,longitude_right,Station name,distance_m,date,temp_3d_lag,wind_3d_max_lag,rain_7d_sum_lag
0,Белгородская область,14000000,14000000,2000,Лесные,к-12 (15254),50.513889,36.653056,Наземный мониторинг,н.п. Разумное,212.0,2.4,Белгородское,2000-05-01,48,2000-05-01,ликвидирован 11.05.2000,48,24,0,0,NaN,NaN,2000-05-11,2000-05-11,10,POINT (36.65306 50.51389),1060043,34009,51.7667,36.1667,KURSK,1.343904,2000-05-01,14.477083,4.125,0.4000
1,Белгородская область,14000000,14000000,2000,Лесные,к-15 (16171),50.911944,37.998889,Наземный мониторинг,н.п. Нов.Безгинка,282.0,8.8,Чернянское,2000-05-04,48,2000-05-04,ликвидирован 14.05.2000,48,11,0,0,NaN,NaN,2000-05-14,2000-05-14,10,POINT (37.99889 50.91194),1069175,34123,51.7000,39.2167,VORONEZ,1.450550,2000-05-04,4.552083,4.750,0.1375
2,Белгородская область,14000000,14000000,2000,Лесные,к-10 (35227),51.053889,36.016944,Наземный мониторинг,н.п. Пены,105.0,7.2,Ивнянское,2000-08-22,48,2000-08-22,ликвидирован 1.09.2000,46,3,0,0,NaN,NaN,2000-09-01,2000-09-01,10,POINT (36.01694 51.05389),1060043,34009,51.7667,36.1667,KURSK,0.728372,2000-08-22,21.981250,2.250,0.0750
3,Белгородская область,14000000,14000000,2000,Лесные,к-48 (35489),50.631944,37.048056,Наземный мониторинг,н.п. Большое Городище,286.0,1.9,Шебекинское,2000-08-23,48,2000-08-23,ликвидирован 2.09.2000,48,10,0,0,NaN,NaN,2000-09-02,2000-09-02,10,POINT (37.04806 50.63194),1060043,34009,51.7667,36.1667,KURSK,1.436822,2000-08-23,24.158333,3.000,0.0750
4,Белгородская область,14000000,14000000,2000,Лесные,к-20 (37323),51.316944,38.233889,Наземный мониторинг,н.п. Старомеловое,214.0,4.8,Горшенченское,2000-09-03,48,2000-09-03,ликвидирован 13.09.2000,5,0,0,0,NaN,NaN,2000-09-13,2000-09-13,10,POINT (38.23389 51.31694),1069175,34123,51.7000,39.2167,VORONEZ,1.054822,2000-09-03,18.406250,2.375,0.1000


In [ ]:
merged_df = merged_df.drop(columns=['oktmo', 'okato', 'code', 'zone_beginning', 'landmark_settlement', 'landmark_azimuth', 'landmark_distance',
                                    'area_beginning', 'current_state', 'area_total', 'area_fund_total', 'area_fund_forest', 'comment', 'zone',
                                    'date_end', 'date_liquidation', 'index_right', 'distance_m', 'Station name'])

KeyError: "['oktmo', 'okato', 'code', 'zone_beginning', 'landmark_settlement', 'landmark_azimuth', 'landmark_distance', 'area_beginning', 'current_state', 'area_total', 'area_fund_total', 'area_fund_forest', 'comment', 'zone', 'date_liquidation', 'index_right', 'Station name'] not found in axis"

In [55]:
merged_df.to_csv('C:/Users/sigmalburg/Documents/МНАД I курс/Пожары!/data_merged.csv', index=False)

In [ ]:
# везде пропусков меньше от 0 до 7304 (максмум) - меньше чем полпроцента
merged_df.isna().sum()

# на техногенных объектах горение может продолжаться месяцами, поэтому оставим только пожары до 30 дней (остается 99,5%)
merged_df = merged_df[merged_df['duration']<30]

In [73]:
merged_df['month'] = pd.to_datetime(merged_df['date_beginning']).dt.month
merged_df['day_of_year'] = pd.to_datetime(merged_df['date_beginning']).dt.dayofyear
merged_df['area_log'] = np.log1p(merged_df['area_forest'])

In [80]:
from sklearn.ensemble import HistGradientBoostingRegressor  # быстрее и устойчивее для таких задач
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
import joblib
import os

os.makedirs('models', exist_ok=True)

# признаки и цель
feature_cols = [
    'month',
    'day_of_year',
    'temp_3d_lag',
    'wind_3d_max_lag',
    'rain_7d_sum_lag'
]

# временное разбиение
train = merged_df[merged_df['date_beginning'].dt.year <= 2021].copy()
val   = merged_df[merged_df['date_beginning'].dt.year == 2022].copy()
test  = merged_df[merged_df['date_beginning'].dt.year >= 2023].copy()

print("Train rows:", len(train), "| Val rows:", len(val), "| Test rows:", len(test))

# Подготовка X и y для двух задач
X_train = train[feature_cols]
X_val   = val[feature_cols]
X_test  = test[feature_cols]

y_train_dur = train['duration']
y_val_dur   = val['duration']
y_test_dur  = test['duration']

y_train_area = train['area_log']
y_val_area   = val['area_log']
y_test_area  = test['area_log']

# --- 3. Модель для длительности (duration) ---
model_dur = HistGradientBoostingRegressor(random_state=42)
model_dur.fit(X_train, y_train_dur)

y_pred_dur_val = model_dur.predict(X_val)
mae_dur_val = mean_absolute_error(y_val_dur, y_pred_dur_val)
rmse_dur_val = np.sqrt(mean_squared_error(y_val_dur, y_pred_dur_val))
print(f"Duration (Val) -> MAE={mae_dur_val:.2f} дней, RMSE={rmse_dur_val:.2f} дней")

# Тест
y_pred_dur_test = model_dur.predict(X_test)
mae_dur_test = mean_absolute_error(y_test_dur, y_pred_dur_test)
print(f"Duration (Test) -> MAE={mae_dur_test:.2f} дней")

print("Duration Val MedianAE:", median_absolute_error(y_val_dur, y_pred_dur_val))

# --- 4. Модель для площади (area_log -> обратно в га) ---
model_area = HistGradientBoostingRegressor(random_state=42)
model_area.fit(X_train, y_train_area)

# Валидация
y_pred_area_log_val = model_area.predict(X_val)
y_pred_area_val = np.expm1(y_pred_area_log_val)  # обратно из log1p
mae_area_val = mean_absolute_error(val['area_forest'], y_pred_area_val)
print(f"Area (Val) -> MAE={mae_area_val:.1f} га")

# Тест
y_pred_area_log_test = model_area.predict(X_test)
y_pred_area_test = np.expm1(y_pred_area_log_test)
mae_area_test = mean_absolute_error(test['area_forest'], y_pred_area_test)
print(f"Area (Test) -> MAE={mae_area_test:.1f} га")

print("Area Val MedianAE (га):", median_absolute_error(val['area_forest'], y_pred_area_val))

# --- 5. Сохранение моделей ---
joblib.dump(model_dur, 'models/duration_model.pkl')
joblib.dump(model_area, 'models/area_model.pkl')

Train rows: 1241028 | Val rows: 69586 | Test rows: 141317
Duration (Val) -> MAE=3.67 дней, RMSE=3.93 дней
Duration (Test) -> MAE=3.77 дней
Duration Val MedianAE: 3.838689885704836
Area (Val) -> MAE=38.8 га
Area (Test) -> MAE=58.9 га
Area Val MedianAE (га): 1.826388999713945


['models/area_model.pkl']

In [15]:
import polars as pl

pl.scan_csv("~/Documents/МНАД I курс/Пожары!/forest_fire_project/data/data_merged.csv").sink_parquet("~/Documents/МНАД I курс/Пожары!/forest_fire_project/data/data_merged.parquet")

In [16]:
prqt = pd.read_parquet("~/Documents/МНАД I курс/Пожары!/forest_fire_project/data/data_merged.parquet")

In [20]:
prqt.columns

Index(['region', 'oktmo', 'okato', 'year', 'type', 'code', 'latitude_left',
       'longitude_left', 'zone_beginning', 'landmark_settlement',
       'landmark_azimuth', 'landmark_distance', 'forestry', 'date_beginning',
       'area_beginning', 'date_end', 'current_state', 'area_total',
       'area_forest', 'area_fund_total', 'area_fund_forest', 'comment', 'zone',
       'date_liquidation', 'date_end_max', 'duration', 'geometry',
       'index_right', 'Синоптический индекс станции', 'latitude_right',
       'longitude_right', 'Station name', 'distance_m', 'date', 'temp_3d_lag',
       'wind_3d_max_lag', 'rain_7d_sum_lag'],
      dtype='str')

In [19]:
dt.columns

Index(['region', 'oktmo', 'okato', 'year', 'type', 'code', 'latitude',
       'longitude', 'zone_beginning', 'landmark_settlement',
       'landmark_azimuth', 'landmark_distance', 'forestry', 'date_beginning',
       'area_beginning', 'date_end', 'current_state', 'area_total',
       'area_forest', 'area_fund_total', 'area_fund_forest', 'comment', 'zone',
       'date_liquidation', 'date_end_max', 'duration'],
      dtype='str')